# 01 — Data Download and Feature Engineering

## Data source note
We use `yfinance` with `interval="1h"` to obtain ~2 years of intraday data.
Yahoo Finance restricts 5-minute bars to the last ~60 days; hourly bars are
available for up to 730 days. For a short validation window you can override
with `interval="5m", start="2024-11-01"`.

## Features
- **Market**: log-returns (1/3/6/12 bars), rolling volatility, relative range,
  volume ratio, time-of-day (sin/cos), Corwin-Schultz bid-ask spread estimate.
- **Order (synthetic)**: side (+1/-1), order_size_fraction, urgency.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # must come before pyplot import; remove for interactive use
import matplotlib.pyplot as plt

from data_loader import TICKERS, download_ohlcv
from features import compute_market_features, add_synthetic_orders, FEATURE_NAMES
from viz import FIGURES_DIR

In [ ]:
# Download (uses parquet cache after first run).
# No explicit dates: defaults use today - 720 days, always within Yahoo's 730-day 1h limit.
data = download_ohlcv(TICKERS, interval='1h')
print(f'Tickers loaded: {list(data.keys())}')

In [ ]:
# Inspect one ticker
spy = data['SPY']
print(spy.tail())
print(f'\nSPY: {len(spy):,} bars  |  date range: {spy.index[0].date()} → {spy.index[-1].date()}')

In [ ]:
# Compute market features for all tickers, each with its own seeded RNG
# (so synthetic-order randomness is independent of dict iteration order).
import hashlib

def _ticker_rng(ticker, base=42):
    h = hashlib.md5(f'{base}:{ticker}'.encode()).hexdigest()
    return np.random.default_rng(int(h[:8], 16) % (2**32))

all_features = {}
for ticker, df in data.items():
    mf = compute_market_features(df)
    feats = add_synthetic_orders(mf, rng=_ticker_rng(ticker))
    all_features[ticker] = feats
    print(f'{ticker}: {len(feats):,} rows after feature engineering')

In [ ]:
# Feature distributions for SPY
# FEATURE_NAMES has 13 entries; use a 4×4 grid (16 slots, last 3 hidden).
n_cols = 4
n_rows = (len(FEATURE_NAMES) + n_cols - 1) // n_cols  # ceil division → 4 rows
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows))
spy_feats = all_features['SPY']
for ax, col in zip(axes.flat, FEATURE_NAMES):
    spy_feats[col].hist(bins=50, ax=ax)
    ax.set_title(col, fontsize=9)
    ax.set_yticks([])
for ax in axes.flat[len(FEATURE_NAMES):]:
    ax.set_visible(False)
plt.suptitle('SPY — Feature Distributions', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Corwin-Schultz spread over time (SPY)
fig, ax = plt.subplots(figsize=(12, 3))
spy_feats['spread_cs'].plot(ax=ax, lw=0.5, alpha=0.7)
ax.set_ylabel('Estimated bid-ask spread (Corwin-Schultz)')
ax.set_title('SPY — Corwin-Schultz Spread Estimate (1h bars)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'corwin_schultz_spy.png', dpi=150)
plt.show()